In [1]:
import numpy as np
import pyvista as pv
from scipy.spatial import KDTree
import sys
sys.path.append("..")
from error_func import error

case_name = "case2"
mesh_name = "coil_box"
solver1 = "ngsolve"
solver2 = "comsol"

sol_ngsolve = pv.read(f"../../output/{case_name}/{case_name}_{solver1}.vtu")
sol_comsol = pv.read(f"../../output/{case_name}/{case_name}_{solver2}.vtu")

In [2]:
tree = KDTree(sol_ngsolve.points)
distances, indices = tree.query(sol_comsol.points)

max_dist = np.max(distances)
print(f"Maximum alignment error (distance): {max_dist:.6e}")
if max_dist > 1e-4:
    print("Warning: Large distance detected. Are the geometries identical?")

aligned_grid = sol_comsol.copy()

for array_name in sol_ngsolve.point_data.keys():
        data = sol_ngsolve.point_data[array_name]
        
        reordered_data = data[indices]
        
        aligned_grid.point_data[array_name] = reordered_data
        print(f"Transferred array: {array_name}")

aligned_grid.save(f"../../output/{case_name}/{case_name}_{solver1}_reordered_to_{solver2}.vtu")

Maximum alignment error (distance): 1.601242e-13
Transferred array: magnetic_vector_potential_nd
Transferred array: magnetic_flux_density_nd
Transferred array: current_density
Transferred array: magnetic_vector_potential
Transferred array: electric_potential
Transferred array: magnetic_flux_density


In [3]:
sol_ngsolve = pv.read(f"../../output/{case_name}/{case_name}_{solver1}_reordered_to_{solver2}.vtu")
elec_pot_ngsolve = sol_ngsolve["electric_potential"]
mag_flux_ngsolve = sol_ngsolve["magnetic_flux_density"]
mag_vec_ngsolve = sol_ngsolve["magnetic_vector_potential"]

sol_comsol = pv.read(f"../../output/{case_name}/{case_name}_{solver2}.vtu")
# elec_pot_comsol = sol_comsol["electric_potential"]
mag_flux_comsol = sol_comsol["magnetic_flux_density"]
mag_vec_comsol = sol_comsol["magnetic_vector_potential"]

# print(elec_pot_ngsolve.shape)
# print(elec_pot_comsol.shape)

print(mag_flux_ngsolve.shape)
print(mag_flux_comsol.shape)

print(mag_vec_ngsolve.shape)
print(mag_vec_comsol.shape)

(278516, 3)
(278516, 3)
(278516, 3)
(278516, 3)


In [4]:
mesh = sol_comsol.copy()

mesh.point_data.remove("magnetic_vector_potential")
mesh.point_data.remove("magnetic_flux_density")

In [5]:
# print(f"Electric potential errors between {solver1} and {solver2}:")

# mesh = error(sol=elec_pot_ngsolve, sol_ref=elec_pot_comsol, 
#              eps = 1e-6, mesh=mesh, tag="scalar", save_tag="V")

In [6]:
print(f"Magnetic flux density errors between {solver1} and {solver2}:")

mesh = error(sol=mag_flux_ngsolve, sol_ref=mag_flux_comsol, 
             eps = 1e-6, mesh=mesh, tag="vector", save_tag="B")

Magnetic flux density errors between ngsolve and comsol:

  * Max. absolute error in x direction  : 8.071e+04.
  * Avg. absolute error in x direction  : 6.211e+03.

  * Max. relative error in x direction : 1.489e+08 %.
  * Avg. relative error in x direction : 1.188e+03 %.

  * Max. absolute error in y direction  : 7.982e+04.
  * Avg. absolute error in y direction  : 4.298e+03.

  * Max. relative error in y direction : 4.199e+06 %.
  * Avg. relative error in y direction : 3.487e+02 %.

  * Max. absolute error in z direction  : 1.122e+05.
  * Avg. absolute error in z direction  : 9.632e+03.

  * Max. relative error in z direction : 3.131e+06 %.
  * Avg. relative error in z direction : 3.186e+02 %.


/home/wiera/Documents/EM_simulation/scripts/case2/../error_func.py:62: RuntimeWarning: divide by zero encountered in divide
  rel_error_dir = np.where(denom > eps, num * 100 / denom, np.nan)


In [7]:
print(f"Magnetic vector potential errors between {solver1} and {solver2}:")

mesh = error(sol=mag_vec_ngsolve, sol_ref=mag_vec_comsol, 
             eps = 1e-6, mesh=mesh, tag="vector", save_tag="A")

Magnetic vector potential errors between ngsolve and comsol:

  * Max. absolute error in x direction  : 1.706e+14.
  * Avg. absolute error in x direction  : 1.000e+12.

  * Max. relative error in x direction : 3.291e+18 %.
  * Avg. relative error in x direction : 2.142e+13 %.

  * Max. absolute error in y direction  : 2.172e+14.
  * Avg. absolute error in y direction  : 1.638e+12.

  * Max. relative error in y direction : 5.913e+16 %.
  * Avg. relative error in y direction : 1.103e+12 %.

  * Max. absolute error in z direction  : 1.436e+14.
  * Avg. absolute error in z direction  : 7.313e+11.

  * Max. relative error in z direction : 3.869e+17 %.
  * Avg. relative error in z direction : 1.439e+13 %.


In [8]:
mesh.save(f"../../output/{case_name}/{case_name}_error_ngsolve_comsol.vtu")